# Dataset y splits

Verifica los pares CT-label `.mha`, construye el dataset curado, revisa la consistencia tecnica de los volumenes y genera los splits fijos por paciente (75/5/20, semilla 42).

- **Entradas:** `data_mha/` (datos locales, no se suben a GitHub).
- **Salidas:** `splits/dataset.csv` (con la columna `split`) y `splits/{train,val,test}.txt`.

Los demas notebooks leen estos archivos en lugar de recalcularlos.

## 1. Librerias base

Se importan las librerias necesarias para trabajar con rutas, tablas, volumenes medicos, arreglos numericos y graficas.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import SimpleITK as sitk

## 2. Configuracion de rutas reproducibles

La raiz del repositorio se localiza subiendo desde `Path.cwd()` hasta encontrar `requirements.txt`. Asi el notebook funciona tanto si el kernel arranca en `notebooks/` como en la raiz, sin rutas absolutas de un computador especifico.

In [2]:
# Raiz del repositorio: funciona si el kernel arranca en notebooks/ o en la raiz.
BASE_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").is_file())
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))  # modulos .py del proyecto

DATA_MHA_DIR = BASE_DIR / "data_mha"

IMAGE_DIRS = [
    DATA_MHA_DIR / "PENGWIN_CT_train_images_part1",
    DATA_MHA_DIR / "PENGWIN_CT_train_images_part2",
]

LABELS_DIR = DATA_MHA_DIR / "PENGWIN_CT_train_labels"
LABEL_DIR = LABELS_DIR  # Alias para compatibilidad con celdas anteriores
SPLITS_DIR = BASE_DIR / "splits"

SPLITS_DIR.mkdir(exist_ok=True)


def to_repo_relative(path):
    """Guarda rutas portables dentro del repositorio."""
    path = Path(path)
    try:
        return path.relative_to(BASE_DIR).as_posix()
    except ValueError:
        return path.as_posix()


def resolve_repo_path(path):
    """Permite leer rutas relativas o absolutas sin romper compatibilidad."""
    path = Path(path)
    if path.is_absolute():
        return path
    return BASE_DIR / path


## 3. Conteo inicial de archivos `.mha`

Se buscan las imagenes CT y las mascaras/labels locales. En esta entrega los datos pesados no se suben a GitHub; cada integrante debe tenerlos en `data_mha/`.

In [3]:
image_files = []

for image_dir in IMAGE_DIRS:
    image_files.extend(image_dir.glob("*.mha"))

image_files = sorted(image_files)
label_files = sorted(LABELS_DIR.glob("*.mha"))

print("RESUMEN DEL DATASET")


print(f"Imágenes encontradas: {len(image_files)}")
print(f"Labels encontrados:   {len(label_files)}")

RESUMEN DEL DATASET
Imágenes encontradas: 100
Labels encontrados:   100


## 4. Extraccion de IDs de casos

Se extraen los IDs desde los nombres de archivo para comparar imagenes y labels sin depender de rutas completas.

In [4]:
image_ids = {
    file.stem
    for file in image_files
}

label_ids = {
    file.stem
    for file in label_files
}

## 5. Validacion de correspondencia imagen-label

Se comprueba que cada imagen tenga su label y que no existan labels sin imagen. Esta validacion es la base del dataset curado.

In [5]:
images_without_labels = sorted(image_ids - label_ids)
labels_without_images = sorted(label_ids - image_ids)

common_ids = sorted(image_ids & label_ids)

print("VALIDACIÓN DE CORRESPONDENCIA")

print(f"Casos con imagen + label: {len(common_ids)}")

print(
    f"Imágenes sin label:       {len(images_without_labels)}"
)

print(
    f"Labels sin imagen:        {len(labels_without_images)}"
)


if images_without_labels:
    print("\nImágenes sin label:")
    for case_id in images_without_labels:
        print(f"  - {case_id}")


if labels_without_images:
    print("\nLabels sin imagen:")
    for case_id in labels_without_images:
        print(f"  - {case_id}")


VALIDACIÓN DE CORRESPONDENCIA
Casos con imagen + label: 100
Imágenes sin label:       0
Labels sin imagen:        0


## 6. Construccion del dataset curado

Se crea una tabla con `case_id`, ruta relativa de imagen y ruta relativa de label. Las rutas se guardan relativas al repositorio para mantener reproducibilidad entre computadores.

In [6]:
dataset = []

for case_id in common_ids:

    # Buscar la imagen en todas las partes
    image_path = None

    for image_dir in IMAGE_DIRS:
        possible_path = image_dir / f"{case_id}.mha"

        if possible_path.exists():
            image_path = possible_path
            break

    # Ruta del label
    label_path = LABELS_DIR / f"{case_id}.mha"

    # Verificar que ambos existan
    if image_path is not None and label_path.exists():

        dataset.append({
            "case_id": case_id,
            "image_path": to_repo_relative(image_path),
            "label_path": to_repo_relative(label_path)
        })


df = pd.DataFrame(dataset)
df


,case_id,image_path,label_path
0,001,data_mha/PENGWIN_CT_train_images_part1/001.mha,data_mha/PENGWIN_CT_train_labels/001.mha
1,002,data_mha/PENGWIN_CT_train_images_part1/002.mha,data_mha/PENGWIN_CT_train_labels/002.mha
2,003,data_mha/PENGWIN_CT_train_images_part1/003.mha,data_mha/PENGWIN_CT_train_labels/003.mha
3,004,data_mha/PENGWIN_CT_train_images_part1/004.mha,data_mha/PENGWIN_CT_train_labels/004.mha
4,005,data_mha/PENGWIN_CT_train_images_part1/005.mha,data_mha/PENGWIN_CT_train_labels/005.mha
...,...,...,...
95,096,data_mha/PENGWIN_CT_train_images_part2/096.mha,data_mha/PENGWIN_CT_train_labels/096.mha
96,097,data_mha/PENGWIN_CT_train_images_part2/097.mha,data_mha/PENGWIN_CT_train_labels/097.mha
97,098,data_mha/PENGWIN_CT_train_images_part2/098.mha,data_mha/PENGWIN_CT_train_labels/098.mha
98,099,data_mha/PENGWIN_CT_train_images_part2/099.mha,data_mha/PENGWIN_CT_train_labels/099.mha


## 7. Exportacion de `dataset.csv`

Se guarda el dataset curado en `splits/dataset.csv`. Este archivo queda versionado en GitHub porque contiene rutas ligeras y reproducibles, no datos pesados.

In [7]:
dataset_path = SPLITS_DIR / "dataset.csv"

df.to_csv(
    dataset_path,
    index=False
)

print()
print(f"Dataset curado guardado en:")
print(to_repo_relative(dataset_path))


Dataset curado guardado en:
splits/dataset.csv


## 8. Verificacion tecnica de volumenes `.mha`

Se cargan imagen y label con SimpleITK para verificar dimensiones, spacing, origen y direccion. Esto confirma que las mascaras corresponden fisicamente con las tomografias.

In [8]:
records = []

for _, row in df.iterrows():

    image = sitk.ReadImage(str(resolve_repo_path(row["image_path"])))
    label = sitk.ReadImage(str(resolve_repo_path(row["label_path"])))

    records.append({
        "case_id": row["case_id"],

        # Imagen
        "image_dimension": image.GetDimension(),
        "image_size": image.GetSize(),
        "image_spacing": image.GetSpacing(),

        # Label
        "label_dimension": label.GetDimension(),
        "label_size": label.GetSize(),
        "label_spacing": label.GetSpacing(),

        # Verificaciones
        "same_size": image.GetSize() == label.GetSize(),
        "same_spacing": image.GetSpacing() == label.GetSpacing(),
        "same_direction": image.GetDirection() == label.GetDirection(),
        "same_origin": image.GetOrigin() == label.GetOrigin()
    })

verification_df = pd.DataFrame(records)

verification_df.head(20)


,case_id,image_dimension,image_size,image_spacing,label_dimension,label_size,label_spacing,same_size,same_spacing,same_direction,same_origin
0,001,3,"(512, 512, 401)","(0.78125, 0.78125, 0.800000011920929)",3,"(512, 512, 401)","(0.78125, 0.78125, 0.800000011920929)",True,True,True,True
1,002,3,"(413, 276, 337)","(0.82421875, 0.82421875, 0.800000011920929)",3,"(413, 276, 337)","(0.82421875, 0.82421875, 0.800000011920929)",True,True,True,True
2,003,3,"(512, 381, 285)","(0.759765625, 0.759765625, 1.0)",3,"(512, 381, 285)","(0.759765625, 0.759765625, 1.0)",True,True,True,True
3,004,3,"(386, 254, 224)","(0.84375, 0.84375, 1.0)",3,"(386, 254, 224)","(0.84375, 0.84375, 1.0)",True,True,True,True
4,005,3,"(512, 512, 312)","(0.896484375, 0.896484375, 1.0)",3,"(512, 512, 312)","(0.896484375, 0.896484375, 1.0)",True,True,True,True
5,006,3,"(426, 279, 283)","(0.8125, 0.8125, 1.0)",3,"(426, 279, 283)","(0.8125, 0.8125, 1.0)",True,True,True,True
6,007,3,"(512, 512, 317)","(0.8359375, 0.8359375, 1.0)",3,"(512, 512, 317)","(0.8359375, 0.8359375, 1.0)",True,True,True,True
7,008,3,"(512, 512, 335)","(0.7421875, 0.7421875, 0.800000011920929)",3,"(512, 512, 335)","(0.7421875, 0.7421875, 0.800000011920929)",True,True,True,True
8,009,3,"(415, 245, 303)","(0.8125, 0.8125, 0.8000000715255737)",3,"(415, 245, 303)","(0.8125, 0.8125, 0.8000000715255737)",True,True,True,True
9,010,3,"(512, 512, 350)","(0.9459999799728394, 0.9459999799728394, 0.798...",3,"(512, 512, 350)","(0.9459999799728394, 0.9459999799728394, 0.798...",True,True,True,True


## 9. Resumen de consistencia del dataset

Se resumen las verificaciones anteriores para confirmar que el dataset local esta completo y alineado.

In [9]:
print("Casos:", len(verification_df))

print("\nDimensiones de las imágenes:")
print(verification_df["image_dimension"].value_counts())

print("\nDimensiones de los labels:")
print(verification_df["label_dimension"].value_counts())

print("\nTamaños CT-label iguales:")
print(verification_df["same_size"].value_counts())

print("\nSpacing CT-label iguales:")
print(verification_df["same_spacing"].value_counts())

Casos: 100

Dimensiones de las imágenes:
image_dimension
3    100
Name: count, dtype: int64

Dimensiones de los labels:
label_dimension
3    100
Name: count, dtype: int64

Tamaños CT-label iguales:
same_size
True    100
Name: count, dtype: int64

Spacing CT-label iguales:
same_spacing
True    100
Name: count, dtype: int64


## 10. Conteo inicial de fragmentos por caso

A partir de las labels se cuenta cuantas etiquetas diferentes aparecen en cada caso, excluyendo el fondo. Esto da una primera idea de la complejidad del dataset.

In [10]:
frag_counts = []

for _, row in df.iterrows():
    label_img = sitk.ReadImage(str(resolve_repo_path(row["label_path"])))
    label_vol = sitk.GetArrayFromImage(label_img)
    n_fragments = len(np.unique(label_vol)) - 1  # excluir fondo (label 0)
    frag_counts.append(n_fragments)

df["n_fragments"] = frag_counts

print("Distribucion de numero de fragmentos por caso:")
print(df["n_fragments"].describe())
print("\nConteo de valores (cuantos casos tienen cada n de fragmentos):")
print(df["n_fragments"].value_counts().sort_index())

Distribucion de numero de fragmentos por caso:
count    100.000000
mean       5.750000
std        1.217507
min        3.000000
25%        5.000000
50%        6.000000
75%        7.000000
max        9.000000
Name: n_fragments, dtype: float64

Conteo de valores (cuantos casos tienen cada n de fragmentos):
n_fragments
3     1
4    13
5    32
6    28
7    18
8     6
9     2
Name: count, dtype: int64


## 11. Splits fijos

Se usa una particion 75/5/20: 75 casos para entrenamiento, 5 para validacion y 20 para test. La semilla fija permite reproducir siempre los mismos splits.

In [11]:
from sklearn.model_selection import train_test_split

SEED = 42

def complexity_bin(n_fragments):
    if n_fragments <= 4:
        return "baja"
    if n_fragments <= 6:
        return "media"
    return "alta"

split_source = df[["case_id", "image_path", "label_path", "n_fragments"]].copy()
split_source["complexity"] = split_source["n_fragments"].apply(complexity_bin)

train_val_df, test_df = train_test_split(
    split_source,
    test_size=20,
    random_state=SEED,
    shuffle=True,
    stratify=split_source["complexity"],
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=5,
    random_state=SEED,
    shuffle=True,
    stratify=train_val_df["complexity"],
)

splits = {
    "train": train_df.sort_values("case_id"),
    "val": val_df.sort_values("case_id"),
    "test": test_df.sort_values("case_id"),
}

for name, split_df in splits.items():
    (SPLITS_DIR / f"{name}.txt").write_text("\n".join(split_df["case_id"].tolist()) + "\n")

split_dataset = []
for name, split_df in splits.items():
    temp = split_df.copy()
    temp.insert(1, "split", name)
    split_dataset.append(temp)

split_dataset = pd.concat(split_dataset, ignore_index=True).sort_values("case_id")
split_dataset.to_csv(SPLITS_DIR / "dataset.csv", index=False)

print("Splits fijos con semilla:", SEED)
print(split_dataset["split"].value_counts().reindex(["train", "val", "test"]))
print("\nDistribucion por complejidad:")
print(split_dataset.groupby(["split", "complexity"]).size().unstack(fill_value=0))

Splits fijos con semilla: 42
split
train    75
val       5
test     20
Name: count, dtype: int64

Distribucion por complejidad:
complexity  alta  baja  media
split                        
test           5     3     12
train         20    10     45
val            1     1      3
